# Introduction to Data-Grounded Agents

A language model **without grounding** answers from parametric memory — weights frozen at training time. It may sound confident while inventing return policies, order statuses, or metric values that never existed in your systems.

A **data-grounded agent** ties every factual claim to **retrieved evidence** from sources you control:

| Source type | Examples | Grounding mechanism |
|-------------|----------|---------------------|
| **Unstructured docs** | Policies, runbooks, wikis | Retrieval (RAG) |
| **Structured data** | Orders, invoices, metrics tables | SQL / API lookup |
| **Live systems** | CRM, shipping tracker | Tool calls |
| **Long documents** | Full contracts in context | Long-context window (when appropriate) |

Grounding is not optional for production assistants that touch **private**, **regulated**, or **fast-changing** data. This notebook introduces the concepts and builds **ShopCo Knowledge Hub** — a support agent that answers from policy documents and an order database, with citations and a grounding evaluation harness.

Everything is self-contained plain Python. No frameworks or prior notebooks are required.

In [ ]:
"""
ShopCo Knowledge Hub — data-grounded agent introduction.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- Unstructured knowledge (policies, FAQs) ---

POLICY_DOCS: list[dict[str, str]] = [
    {
        "id": "pol-returns-01",
        "title": "Returns Policy",
        "text": "Customers may return items within 30 days of delivery. Items must be unused and in original packaging. Refunds are processed within 5-7 business days after the warehouse receives the return. Final-sale items cannot be returned.",
    },
    {
        "id": "pol-shipping-02",
        "title": "Shipping Policy",
        "text": "Standard shipping is free on orders over $50. Express shipping is available at checkout. Tracking numbers are emailed when the carrier scans the package.",
    },
    {
        "id": "pol-billing-03",
        "title": "Billing Policy",
        "text": "Charges appear as SHOPCO* on credit card statements. Partial refunds may take up to two billing cycles to appear. Contact support with your order ID for billing disputes.",
    },
    {
        "id": "faq-tracking-04",
        "title": "Tracking FAQ",
        "text": "If your order status is 'shipped', use the tracking number from your confirmation email. Processing orders do not yet have tracking numbers.",
    },
]

# --- Structured knowledge (orders table simulation) ---

ORDERS_DB: list[dict[str, Any]] = [
    {
        "order_id": "ORD-1001",
        "customer_email": "alice@example.com",
        "status": "shipped",
        "total_usd": 89.50,
        "tracking": "1Z999AA10123456784",
        "shipped_at": "2026-07-01",
    },
    {
        "order_id": "ORD-1002",
        "customer_email": "bob@example.com",
        "status": "processing",
        "total_usd": 42.00,
        "tracking": None,
        "shipped_at": None,
    },
    {
        "order_id": "ORD-1003",
        "customer_email": "carol@example.com",
        "status": "delivered",
        "total_usd": 120.00,
        "tracking": "1Z999AA10987654321",
        "shipped_at": "2026-06-15",
    },
]

print(f"Corpus: {len(POLICY_DOCS)} policy docs, {len(ORDERS_DB)} orders")

---

## 1. Why Grounding Matters

Ungrounded models optimize for **plausible text**, not **truth in your systems**.

| Risk | Ungrounded behavior | Grounded behavior |
|------|---------------------|-------------------|
| **Hallucination** | Invents a 60-day return window | Cites `pol-returns-01`: 30 days |
| **Stale knowledge** | Answers from 2024 training cutoff | Reads current policy doc version |
| **Private data leak** | Guesses order status | Queries `ORDERS_DB` for exact row |
| **No audit trail** | "The model said so" | Citations + query log for compliance |

**Data-grounded agents** add a retrieve-then-answer (or query-then-answer) step so the model reasons **over evidence**, not imagination.

---

## 2. Grounding Mechanisms at a Glance

```
                    ┌─────────────────────────────────────┐
  user question ───►│      Data-Grounded Agent           │
                    │                                     │
                    │  ┌─────────┐ ┌─────────┐ ┌───────┐ │
                    │  │   RAG   │ │SQL/Table│ │ Tools │ │
                    │  │  index  │ │ lookup  │ │ / API │ │
                    │  └────┬────┘ └────┬────┘ └───┬───┘ │
                    │       └───────────┼──────────┘     │
                    │                   ▼                │
                    │         evidence bundle + cites    │
                    │                   ▼                │
                    │              LLM reasoning         │
                    └───────────────────┬─────────────────┘
                                        ▼
                           grounded answer + provenance
```

Production agents often **combine** mechanisms: RAG for policy prose, SQL for order rows, tools for carrier tracking APIs. Vector indexes, SQL guardrails, and graph-based orchestration are natural extensions of the stack introduced here.

---

## 3. Evidence Types and When to Use Each

| Mechanism | Best for | Weakness |
|-----------|----------|----------|
| **RAG (retrieval)** | Prose policies, runbooks, long docs | Chunking misses context; wrong chunks mislead |
| **SQL / structured query** | Exact counts, filters, joins | Needs schema grounding; SQL injection risk |
| **Tool / API call** | Live status, side effects | Latency; requires auth and idempotency |
| **Long context** | Small stable corpora, single PDF | Cost scales with tokens; no fresh DB data |
| **Fine-tuning** | Style, format, domain jargon | Does not inject live facts at runtime |

Production agents often **combine** mechanisms: RAG for policy prose, SQL for order rows, tools for carrier tracking APIs.

---

## 4. Grounded Answer Contract

A grounded response is not just text — it carries **provenance**:

- `answer` — natural language for the user
- `citations` — doc IDs or table/query references
- `evidence` — raw snippets the model saw
- `grounded` — whether any evidence was attached
- `abstained` — whether the agent refused due to missing evidence

In [ ]:
class EvidenceSource(str, Enum):
    RAG = "rag"
    SQL = "sql"
    TOOL = "tool"
    NONE = "none"


@dataclass
class Citation:
    source: EvidenceSource
    ref: str
    snippet: str


@dataclass
class GroundedAnswer:
    answer: str
    citations: list[Citation] = field(default_factory=list)
    grounded: bool = False
    abstained: bool = False
    trace: list[str] = field(default_factory=list)

    def citation_ids(self) -> list[str]:
        return [c.ref for c in self.citations]


sample = GroundedAnswer(
    answer="Returns are allowed within 30 days.",
    citations=[Citation(EvidenceSource.RAG, "pol-returns-01", "30 days of delivery")],
    grounded=True,
)
print(pretty(sample))

---

## 5. Document Retriever (RAG Layer)

Keyword overlap over a small corpus is enough to teach the control flow. Production systems swap this for embedding search (FAISS, pgvector, etc.).

In [ ]:
def retrieve_docs(query: str, top_k: int = 2) -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored: list[tuple[int, dict[str, str]]] = []
    for doc in POLICY_DOCS:
        haystack = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for term in terms if term in haystack) if terms else 0
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda x: (-x[0], x[1]["id"]))
    return [{"id": d["id"], "title": d["title"], "text": d["text"]} for _, d in scored[:top_k]]


TEST_QUERIES = [
    "What is the return window?",
    "How long do refunds take?",
    "When do I get a tracking number?",
]

for q in TEST_QUERIES:
    hits = retrieve_docs(q)
    print(f"Q: {q}")
    print(f"  → {[h['id'] for h in hits]}\n")

---

## 6. Structured Lookup (SQL Layer)

For tabular facts — order status, totals, dates — retrieval over prose is unreliable. A **structured query** returns exact rows.

We simulate SQL with a safe, parameterized lookup function. Full text-to-SQL agents come later in this module.

In [ ]:
def lookup_order(order_id: str) -> dict[str, Any] | None:
    """Parameterized order lookup — stand-in for SELECT * FROM orders WHERE order_id = ?"""
    if not re.match(r"^ORD-[0-9]{4}$", order_id):
        return None
    for row in ORDERS_DB:
        if row["order_id"] == order_id:
            return dict(row)
    return None


def simulate_sql_query(intent: str, order_id: str | None = None) -> dict[str, Any]:
    """Route natural-language intent to safe structured queries only."""
    intent_l = intent.lower()
    if order_id:
        row = lookup_order(order_id)
        if not row:
            return {"query": f"orders WHERE order_id='{order_id}'", "rows": [], "found": False}
        return {"query": f"orders WHERE order_id='{order_id}'", "rows": [row], "found": True}
    if "how many" in intent_l and "order" in intent_l:
        return {"query": "SELECT COUNT(*) FROM orders", "rows": [{"count": len(ORDERS_DB)}], "found": True}
    return {"query": "(unsupported)", "rows": [], "found": False}


print(pretty(lookup_order("ORD-1001")))
print(pretty(simulate_sql_query("status check", "ORD-1002")))

---

## 7. Ungrounded Baseline — What Goes Wrong

A mock ungrounded model returns plausible but **uncited** answers. Useful as a regression baseline in eval harnesses.

In [ ]:
def ungrounded_responder(question: str) -> GroundedAnswer:
    """Simulates an LLM answering without retrieval — may hallucinate."""
    q = question.lower()
    if "return" in q:
        # Wrong: hallucinated 60-day window
        return GroundedAnswer(
            answer="You can return items within 60 days for a full refund.",
            grounded=False,
            trace=["parametric_memory_only"],
        )
    if "ord-1001" in q:
        return GroundedAnswer(
            answer="Order ORD-1001 is still processing.",
            grounded=False,
            trace=["parametric_memory_only"],
        )
    return GroundedAnswer(
        answer="I'm not sure. Please contact support.",
        grounded=False,
        trace=["parametric_memory_only"],
    )


bad = ungrounded_responder("What is your return policy?")
print("Ungrounded:", bad.answer)
print("Citations:", bad.citation_ids())

---

## 8. Grounded Responder — Retrieve Then Answer

The grounded path:

1. Classify question type (policy vs order status)
2. Fetch evidence (RAG or SQL)
3. Compose answer **only from evidence**
4. Attach citations; abstain if evidence is empty

In [ ]:
def classify_question(question: str) -> Literal["policy", "order", "mixed", "unknown"]:
    q = question.lower()
    has_order = bool(re.search(r"ord-[0-9]{4}", q))
    policy_terms = ("return", "refund", "shipping", "billing", "track", "policy")
    has_policy = any(t in q for t in policy_terms)
    if has_order and has_policy:
        return "mixed"
    if has_order:
        return "order"
    if has_policy:
        return "policy"
    return "unknown"


def compose_from_evidence(question: str, citations: list[Citation]) -> str:
    """Offline composer — in production, LLM synthesizes with cite-or-abstain prompt."""
    if not citations:
        return "I don't have enough verified information to answer that."
    q = question.lower()
    if "return" in q or "refund" in q:
        for c in citations:
            if "30 days" in c.snippet:
                return f"You may return items within 30 days of delivery. Refunds take 5-7 business days. [{c.ref}]"
    if "ord-" in q:
        for c in citations:
            if c.source == EvidenceSource.SQL and "shipped" in c.snippet:
                return f"Your order is shipped. Tracking details are in our records. [{c.ref}]"
            if c.source == EvidenceSource.SQL and "processing" in c.snippet:
                return f"Your order is still processing — no tracking number yet. [{c.ref}]"
    top = citations[0]
    return f"Based on ShopCo records: {top.snippet[:120]}... [{top.ref}]"


def grounded_responder(question: str) -> GroundedAnswer:
    trace: list[str] = []
    qtype = classify_question(question)
    trace.append(f"classify:{qtype}")
    citations: list[Citation] = []

    if qtype in ("policy", "mixed", "unknown"):
        docs = retrieve_docs(question, top_k=2)
        trace.append(f"rag:hits={len(docs)}")
        for doc in docs:
            citations.append(Citation(EvidenceSource.RAG, doc["id"], doc["text"]))

    if qtype in ("order", "mixed"):
        m = re.search(r"ORD-[0-9]{4}", question.upper())
        if m:
            oid = m.group(0)
            result = simulate_sql_query(question, oid)
            trace.append(f"sql:found={result['found']}")
            if result["found"]:
                row = result["rows"][0]
                citations.append(
                    Citation(EvidenceSource.SQL, result["query"], pretty(row))
                )

    if not citations:
        return GroundedAnswer(
            answer="I cannot verify an answer without matching ShopCo records. Please rephrase or provide an order ID.",
            abstained=True,
            grounded=False,
            trace=trace,
        )

    answer = compose_from_evidence(question, citations)
    return GroundedAnswer(
        answer=answer,
        citations=citations,
        grounded=True,
        trace=trace,
    )


good = grounded_responder("What is your return policy?")
print("Grounded:", good.answer)
print("Citations:", good.citation_ids())

---

## 9. Side-by-Side — Ungrounded vs Grounded

In [ ]:
COMPARE_QUESTIONS = [
    "What is your return policy?",
    "What is the status of ORD-1001?",
    "Can I return after 45 days?",
]

print(f"{'Question':<42} {'Mode':<12} Grounded  Citations")
print("-" * 85)
for q in COMPARE_QUESTIONS:
    for label, fn in [("ungrounded", ungrounded_responder), ("grounded", grounded_responder)]:
        out = fn(q)
        cites = ",".join(out.citation_ids()) or "—"
        print(f"{q[:40]:<42} {label:<12} {str(out.grounded):<9} {cites}")

---

## 10. Grounding in an Agent Loop

A **single-shot** RAG call is level 1. A **data-grounded agent** may retrieve multiple times, query SQL, validate evidence, and only then answer.

```
  START
    │
    ▼
 classify intent
    │
    ├── policy ──► RAG retrieve ──┐
    ├── order  ──► SQL lookup  ──┼──► evidence merge
    └── mixed  ──► both        ──┘
                    │
                    ▼
              sufficient evidence?
               /              \
             yes               no → abstain / ask clarifying Q
              │
              ▼
         compose + cite
              │
              ▼
            END
```

In [ ]:
@dataclass
class AgentState:
    question: str
    qtype: str = ""
    evidence: list[Citation] = field(default_factory=list)
    status: str = "running"
    trace: list[str] = field(default_factory=list)


class DataGroundedAgent:
    """Minimal multi-step grounding loop — no framework required."""

    def run(self, question: str) -> GroundedAnswer:
        state = AgentState(question=question)
        state.qtype = classify_question(question)
        state.trace.append(f"step:classify→{state.qtype}")

        if state.qtype in ("policy", "mixed", "unknown"):
            for doc in retrieve_docs(question):
                state.evidence.append(Citation(EvidenceSource.RAG, doc["id"], doc["text"]))
            state.trace.append(f"step:rag→{len(state.evidence)} chunks")

        if state.qtype in ("order", "mixed"):
            m = re.search(r"ORD-[0-9]{4}", question.upper())
            if m:
                result = simulate_sql_query(question, m.group(0))
                if result["found"]:
                    state.evidence.append(
                        Citation(EvidenceSource.SQL, result["query"], pretty(result["rows"][0]))
                    )
                state.trace.append(f"step:sql→found={result['found']}")

        if not state.evidence:
            state.status = "abstained"
            return GroundedAnswer(
                answer="I could not find verified ShopCo data for this question.",
                abstained=True,
                trace=state.trace,
            )

        state.status = "answered"
        return GroundedAnswer(
            answer=compose_from_evidence(question, state.evidence),
            citations=state.evidence,
            grounded=True,
            trace=state.trace,
        )


agent = DataGroundedAgent()
mixed = agent.run("I want to return ORD-1003 — what is the policy and order status?")
print("Answer:", mixed.answer)
print("Evidence sources:", [c.source.value for c in mixed.citations])
print("Trace:", mixed.trace)

---

## 11. Citation Enforcement

Grounding fails silently if the model cites documents it never retrieved. Enforce:

1. Every `[doc-id]` in the answer must appear in `citations`
2. Answers with factual claims but zero citations flag as `grounded=False`

In [ ]:
CITE_PATTERN = re.compile(r"\[([a-z0-9_-]+)\]", re.I)


def validate_citations(answer: GroundedAnswer) -> list[str]:
    issues: list[str] = []
    cited_in_text = set(CITE_PATTERN.findall(answer.answer))
    allowed = set(answer.citation_ids())
    for cite in cited_in_text:
        if cite not in allowed:
            issues.append(f"phantom citation: [{cite}] not in evidence bundle")
    if answer.grounded and not answer.citations:
        issues.append("marked grounded but citations list is empty")
    if not answer.grounded and cited_in_text:
        issues.append("ungrounded answer contains citation markers")
    return issues


valid = grounded_responder("What is your return policy?")
invalid = GroundedAnswer(
    answer="Returns allowed [pol-fake-99].",
    citations=[Citation(EvidenceSource.RAG, "pol-returns-01", "30 days")],
    grounded=True,
)

print("Valid issues:", validate_citations(valid))
print("Invalid issues:", validate_citations(invalid))

---

## 12. Abstention When Evidence Is Weak

A grounded agent should **refuse** rather than guess when retrieval returns nothing relevant. Abstention is a feature, not a failure.

In [ ]:
ABSTAIN_CASES = [
    "What is the capital of France?",
    "Status of order ORD-9999?",
    "Tell me a joke about databases.",
]

for q in ABSTAIN_CASES:
    out = grounded_responder(q)
    print(f"Q: {q}")
    print(f"  abstained={out.abstained}  grounded={out.grounded}")
    print(f"  → {out.answer[:70]}...\n")

---

## 13. Grounding Evaluation Harness

Measure grounding with a **golden set**: each row has `expect_citation` or `expect_fact` substrings that must appear in evidence or answer.

In [ ]:
GOLDEN_SET: list[dict[str, Any]] = [
    {
        "question": "What is your return policy?",
        "expect_citation": "pol-returns-01",
        "expect_fact": "30 days",
    },
    {
        "question": "Status of ORD-1001?",
        "expect_citation": "orders WHERE",
        "expect_fact": "shipped",
    },
    {
        "question": "Status of ORD-1002?",
        "expect_fact": "processing",
    },
    {
        "question": "What is quantum entanglement?",
        "expect_abstain": True,
    },
]


def eval_grounding(answer: GroundedAnswer, row: dict[str, Any]) -> bool:
    if row.get("expect_abstain"):
        return answer.abstained or not answer.grounded
    cite_ok = True
    if "expect_citation" in row:
        blob = " ".join(answer.citation_ids()) + " " + answer.answer
        cite_ok = row["expect_citation"] in blob
    fact_ok = True
    if "expect_fact" in row:
        fact_ok = row["expect_fact"].lower() in answer.answer.lower() or any(
            row["expect_fact"].lower() in c.snippet.lower() for c in answer.citations
        )
    return cite_ok and fact_ok and answer.grounded


def run_eval(responder) -> float:
    hits = sum(1 for row in GOLDEN_SET if eval_grounding(responder(row["question"]), row))
    return hits / len(GOLDEN_SET)


ungrounded_score = run_eval(ungrounded_responder)
grounded_score = run_eval(grounded_responder)

print(f"Ungrounded eval: {ungrounded_score:.0%}")
print(f"Grounded eval:   {grounded_score:.0%}")

---

## 14. Multi-Source Grounding Preview

Real tickets often need **both** prose policies and live order rows. The agent merges evidence from multiple sources into one bundle before synthesis.

In [ ]:
def merge_evidence(*bundles: list[Citation]) -> list[Citation]:
    seen: set[str] = set()
    merged: list[Citation] = []
    for bundle in bundles:
        for c in bundle:
            key = f"{c.source}:{c.ref}"
            if key not in seen:
                seen.add(key)
                merged.append(c)
    return merged


q = "Can I return ORD-1003? It was delivered June 15."
rag_cites = [Citation(EvidenceSource.RAG, d["id"], d["text"]) for d in retrieve_docs(q)]
sql_result = simulate_sql_query(q, "ORD-1003")
sql_cites = [Citation(EvidenceSource.SQL, sql_result["query"], pretty(sql_result["rows"][0]))]
merged = merge_evidence(rag_cites, sql_cites)

print(f"Merged {len(merged)} evidence items:")
for c in merged:
    print(f"  [{c.source.value}] {c.ref}")

---

## 15. Data Freshness and Versioning

Grounding quality depends on **fresh indexes** and **versioned documents**.

| Concern | Practice |
|---------|----------|
| Stale RAG index | Re-embed on doc publish; track `doc_version` in metadata |
| SQL drift | Schema registry; validate generated SQL against allowlist |
| Policy changes | Citation includes version: `[pol-returns-01@v3]` |
| Cache invalidation | TTL on retrieved chunks; bypass cache for order status |

In [ ]:
@dataclass
class VersionedDoc:
    doc: dict[str, str]
    version: str
    indexed_at: str


INDEX: list[VersionedDoc] = [
    VersionedDoc(doc=d, version="v1", indexed_at=datetime.now(timezone.utc).isoformat())
    for d in POLICY_DOCS
]

def cite_with_version(citation: Citation, version: str = "v1") -> str:
    return f"[{citation.ref}@{version}]"


c = Citation(EvidenceSource.RAG, "pol-returns-01", "30 days")
print("Versioned cite:", cite_with_version(c))

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| RAG for everything | Wrong order status from prose | SQL/tools for structured facts |
| No citations | Cannot audit answers | `GroundedAnswer` + cite-or-abstain prompt |
| Retrieve after answer | Hallucination already committed | Retrieve **before** synthesis |
| One-shot only | Complex questions miss evidence | Multi-step agent loop |
| Ignoring abstention | Confident wrong answers | Golden set with `expect_abstain` |
| Trusting chunk text blindly | Prompt injection in docs | Sanitize; separate instructions from data |

---

## 17. Extending the Grounding Stack

The minimal stack in this notebook scales along several axes:

| Extension | What it adds |
|-----------|-------------|
| **Vector retrieval** | Semantic search over embeddings (FAISS, pgvector) |
| **Hybrid search** | Keyword + vector + re-ranking for higher recall |
| **Text-to-SQL** | Natural language → validated queries over warehouses |
| **Agentic RAG** | Re-retrieve when first pass evidence is insufficient |
| **Graph orchestration** | Retrieval nodes inside LangGraph with checkpoints |
| **Mixed sources** | Merge RAG chunks and SQL rows in one evidence bundle |

Regardless of mechanism, keep the same contract: evidence bundle → validate → cite → answer or abstain.

---

## 18. Optional Live LLM with Retrieved Context

Set `USE_LIVE_LLM = True` to synthesize answers with `gpt-4o-mini` using retrieved chunks in the prompt. Retrieval stays deterministic; only composition uses the model.

In [ ]:
def live_grounded_answer(question: str) -> GroundedAnswer:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage

    docs = retrieve_docs(question, top_k=3)
    m = re.search(r"ORD-[0-9]{4}", question.upper())
    sql_cites: list[Citation] = []
    if m:
        result = simulate_sql_query(question, m.group(0))
        if result["found"]:
            sql_cites.append(Citation(EvidenceSource.SQL, result["query"], pretty(result["rows"][0])))

    rag_cites = [Citation(EvidenceSource.RAG, d["id"], d["text"]) for d in docs]
    all_cites = merge_evidence(rag_cites, sql_cites)
    if not all_cites:
        return GroundedAnswer(answer="Cannot verify.", abstained=True)

    evidence_block = "\n\n".join(f"[{c.ref}] {c.snippet}" for c in all_cites)
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    response = llm.invoke([
        SystemMessage(content="Answer ONLY from evidence. Cite doc ids in brackets. If insufficient, say you cannot verify."),
        HumanMessage(content=f"Question: {question}\n\nEvidence:\n{evidence_block}"),
    ])
    return GroundedAnswer(
        answer=response.content,
        citations=all_cites,
        grounded=True,
        trace=["live_llm_compose"],
    )


if USE_LIVE_LLM:
    live = live_grounded_answer("What is the return policy for ShopCo?")
    print(live.answer)
else:
    print("Offline mode. Grounded agent eval:", f"{grounded_score:.0%}")
    print("Set USE_LIVE_LLM = True to compose answers with retrieved evidence via gpt-4o-mini.")

---

## 19. Quiz

1. What is the difference between a chatbot with RAG and a **data-grounded agent**?
2. When should you use SQL lookup instead of document retrieval?
3. What fields belong in a `GroundedAnswer` provenance contract?
4. Why is abstention preferable to a confident hallucination?
5. How does the golden-set eval detect ungrounded regressions?

---

## 20. Summary

**Data-grounded agents** connect language models to evidence you trust — policy documents, database rows, APIs — before they speak. Grounding reduces hallucination, enables audit trails via citations, and supports abstention when evidence is missing.

ShopCo Knowledge Hub demonstrated:

- **RAG** for unstructured policies
- **SQL-style lookup** for order facts
- **GroundedAnswer** with citations and trace
- **Eval harness** comparing ungrounded vs grounded quality

The retrieval mechanism can grow (FAISS, hybrid search, text-to-SQL), but the contract stays the same: **retrieve evidence → validate → cite → answer or abstain**.